# Session 17: Prompt Evaluation

##  A Golden Set, a Rubric, and an LLM Judge

In this session, I explore how to evaluate prompts and compare their performance. I use a golden set of test cases, a rubric to define what a good response looks like, and an LLM judge to assess the model's outputs.

## Notebook Structure

| Part | What I will cover |
|---|---|
| **Part A: Concepts** | Understand prompt evaluation and the key terms used in this session. |
| **Part B: Experiments** | Complete Sections 1–9 and test the evaluation methods using code. |
| **Part C: Review** | Summarize the main concepts, review interview questions, and troubleshoot common issues. |

## How I Work Through the Experiments

Each hands-on section follows a simple process:

1. **Purpose:** Understand what I am testing and why it matters.
2. **Concept:** Learn the main idea using a simple example or diagram.
3. **Code Overview:** Understand what the code is expected to do.
4. **Experiment:** Run the code and examine the results.
5. **Results Analysis:** Interpret the output and investigate unexpected results.
6. **Self-Check:** Review my understanding and identify common mistakes.

Optional explanations are available when I need more detail.

By the end of this session, I aim to understand how to compare prompt versions systematically, evaluate responses against clear criteria, and identify where a prompt needs improvement.

---

# Part A: Understanding Prompt Evaluation

## A1. Why This Session Matters

### The Problem

In Session 16, I compared two prompts using customer-support tickets. The structured classifier got 18 out of 20 tickets correct, while the baseline got 13 out of 20.

However, accuracy alone does not tell me everything about a response. For example, a customer-support reply might have the correct label but still be impolite, provide incorrect refund information, or use the wrong format.

To evaluate prompts properly, I need to check more than just the final label. I need a consistent way to compare responses and identify mistakes.

In this session, I focus on three things:

- **Golden Set:** A fixed set of test cases used to compare prompt versions. I reuse the held-out tickets from Session 16.
- **Rubric:** A set of clear criteria used to judge whether a response meets the requirements.
- **LLM-as-Judge:** A second model that evaluates a response using the rubric and returns a structured result.

### What I Want to Learn

| Technique | Why I use it |
|---|---|
| Create a rubric | To define evaluation criteria before checking the responses. |
| Use an LLM judge | To evaluate responses in a consistent, structured format. |
| Understand judge bias | To recognize that a judge model can also make mistakes. |
| Shuffle answer order | To reduce the risk that answer position affects the judge's decision. |
| Check results manually | To compare the judge's decisions with human judgement. |

### Where I Can Use These Techniques

- **Banking:** Check whether AI-generated customer-support replies follow approved policies.
- **Healthcare:** Evaluate summaries against clearly defined review criteria.
- **Retail and insurance:** Compare prompt versions before using them in real applications.
- **AI development teams:** Track prompt quality and identify problems when prompts change.



## A2. Key Terms for Prompt Evaluation

### 1. Golden Set

A **golden set** is a fixed set of test cases used to compare different prompt versions.

**Example:** I reuse the same 20 held-out tickets from Session 16 to evaluate both v1 and v2. The tickets and expected labels remain unchanged.

**Why it matters:** It helps me compare prompts fairly using the same test data.

### 2. Rubric

A **rubric** is a list of clear criteria used to evaluate a response. I define the criteria before checking the results.

**Example:** I check whether the response has the correct label, follows the required rules, uses the expected format, and avoids inventing policies.

**Why it matters:** It makes evaluation more consistent and reduces judgement based only on personal opinion.

### 3. LLM-as-Judge

An **LLM-as-judge** uses a second model call to evaluate the response generated by the first model.

**Example:** `judge_answer()` sends the ticket, the generated response, and the rubric to a model. The judge returns a PASS or FAIL for each criterion.

**Why it matters:** It helps evaluate many responses without manually reviewing every result. However, the judge can also make mistakes.

### 4. Strict JSON Verdict

A **strict JSON verdict** is a result returned in a fixed format so that Python can process and count the scores.

**Example:**

```json
{
  "c1": "PASS",
  "c2": "FAIL",
  "c3": "PASS",
  "c4": "PASS"
}
```

Each key represents a criterion, and its value must be either `PASS` or `FAIL`.

**Why it matters:** Structured results are easier to validate and calculate than free-text explanations.

### 5. Position Bias

**Position bias** occurs when a judge prefers an answer because of its position rather than its actual quality.

**Example:** If I always present v1 before v2, the judge might favor the first answer.

**How I reduce it:** I vary the answer order across test cases and check whether the comparison results remain consistent.

### 6. Verbosity Bias

**Verbosity bias** occurs when a judge prefers longer answers even when a shorter answer is more accurate.

**Example:** A long response with the wrong label might receive a better evaluation than a short response with the correct label.

**How I reduce it:** I use specific PASS/FAIL criteria instead of asking the judge to give a general quality score. This can reduce verbosity bias but cannot eliminate it completely.

### 7. Self-Preference Bias

**Self-preference bias** occurs when a judge favors responses that resemble its own usual writing or reasoning style.

**Example:** The answer generator and judge in this experiment use the same model family, so the judge may favor certain styles of response.

**How I address it:** I recognize this limitation and consider testing with a different, preferably stronger, judge model in future experiments.

### 8. Order Guard

An **order guard** changes which prompt is evaluated first for each test case to reduce ordering bias.

**Example:** `shuffle_order()` determines whether v1 or v2 goes first for a particular ticket.

The order is determined using a consistent method, so the same ticket gets the same order when the experiment is repeated.

**Why it matters:** It reduces fixed-order bias while keeping the experiment reproducible.

### 9. Human Spot-Check

A **human spot-check** means manually evaluating a small sample of responses using the same rubric as the LLM judge.

**Example:** I manually evaluate five tickets and compare my results with the judge's decisions.

**Why it matters:** It helps identify mistakes made by the judge. Checking five tickets provides useful evidence, but it does not guarantee that every judgement is correct.

### 10. Pass Rate

A **pass rate** is the percentage of evaluated responses that pass a particular criterion or all criteria together.

**Example:** If 16 out of 20 responses pass the correct-label criterion:

\[
\text{Pass Rate}=\frac{16}{20}\times100=80\%
\]



## 1. Setup

I loaded the required helper files, model settings, cache, and token tracker for Session 17. I also configured the notebook to access my Session 16 prompt files so I can compare the baseline and structured prompts using the same test cases.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                              # Session 11
    from reasoning_utils import DiskCache, Lab, Meter                                     # Session 15
    from structured_tasks import GUIDE, HELDOUT, LABELS, REFERENCE_SYSTEM, REFERENCE_USER # Session 16
    from structured_utils import SUPPORT_DIR, SYSTEM_FILE, USER_FILE, load_prompt_file, pick_examples  # Session 16
    from evaluation_tasks import (DEFAULT_FORMAT_RULE, GOLDEN20, JUDGE_SYSTEM, JUDGE_USER, RUBRIC, RUBRIC_IDS,  # today
                                  SAMPLE_REPLY_A, SAMPLE_REPLY_B, SAMPLE_TICKET, V1_INSTRUCTION)
    from evaluation_utils import (JudgeParseError, S17_CACHE, agreement, agreement_rate, answer_with,           # today
                                  check_s17_work, compare_prompts, escape_judge_tags, eval_markdown, judge_answer,
                                  judge_messages, parse_eval_report, parse_spot_check, parse_verdict, pass_rates,
                                  pct, save_eval_report, score_prompt, shuffle_order, spot_check_items, v1_messages,
                                  v2_messages_from_files)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14, 15, 16 and today: run 'git pull' in the "
                      "course folder, then restart the kernel.") from e

MODEL = os.getenv("MODEL")
WORK_REPO = os.getenv("WORK_REPO")
cache, meter = DiskCache(S17_CACHE), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None     # one Lab plays BOTH roles today: answerer and judge
ready = lab is not None

print(f"model: {MODEL} | ready: {ready}")
print(f"answers are remembered in: {S17_CACHE}")
print(f"golden set: {len(GOLDEN20)} tickets · rubric: {len(RUBRIC)} criteria {RUBRIC_IDS}")
print(f"WORK_REPO: {WORK_REPO or '(not set — section 6 will need it)'}")
if not ready:
    print("No key: sections 4-6 print 'skipped'. Sections 2, 3, 7, 8 need no model and still run.")

### Expected Output

The notebook shows the model status, cache location, golden set size, rubric criteria, and repository path.

If something is missing, I fix it before continuing.

### Check Yourself

**Question:** Why are the answerer and judge separate calls?

**Answer:** The answerer generates the response, and the judge checks it.

**Common Mistake:** Using the old Session 16 cache may return old results.

## 2. Comparing Two Replies

### Purpose
I compare two replies to the same customer ticket to understand which one is better.

### My Approach
I check whether each reply has the correct label, follows the required format, and avoids making unsupported promises.

### My Observation
Two replies can have the correct label but differ in quality. I will compare both replies with the LLM judge's results later in this session.

In [ ]:
print("TICKET:", SAMPLE_TICKET)
print()
print("REPLY A:", SAMPLE_REPLY_A)
print()
print("REPLY B:", SAMPLE_REPLY_B)

### What I Observe

Reply A has the correct label for the UPI transfer issue. Reply B uses the wrong label and mentions a 24-hour cooling-off period that is not part of the bank's rules.

### My Observation

A reply can be wrong in more than one way. I need to check the label, the information provided, and whether the reply follows the bank's rules.

**Common Mistake:** Checking only the label and ignoring incorrect information or promises in the reply.

## 3. Golden Set and Rubric

### Purpose
I use a fixed set of 20 tickets and define clear criteria before evaluating the replies. This helps me compare prompts fairly.

### My Approach
- **Golden Set:** I reuse the same 20 held-out tickets from Session 16.
- **Rubric:** I use four criteria to check the quality of each reply.

The data stays the same, but the evaluation changes from checking only label accuracy to checking four aspects of the reply.

### What I Check in the Code
1. `GOLDEN20 is HELDOUT` checks whether both variables refer to the same Python object.
2. The code displays the four rubric criteria so I can review them before using the LLM judge.

### My Observation
I define the test data and evaluation criteria before judging the replies. This helps keep the evaluation consistent and reduces bias.

In [ ]:
print("GOLDEN20 is Session 16's HELDOUT:", GOLDEN20 is HELDOUT)
print(f"{len(GOLDEN20)} tickets, 5 per label: {LABELS}")
print()
for c in RUBRIC:
    print(f"{c['id']}. {c['name']}")
    print(f"   {c['test']}")

### Expected Output

The code confirms that `GOLDEN20` uses the same held-out data from Session 16. It also displays the 20 tickets and the four rubric criteria:

- **c1:** Correct label
- **c2:** Correct house rule
- **c3:** Correct reply format
- **c4:** No invented policy

This step does not require a model or API key.

### My Observation

Using the same 20 tickets keeps the evaluation consistent. Creating a new list could change the test data and make the comparison less fair.

The four criteria help me evaluate the full reply, not just its label.

## 4. Testing the LLM Judge

### Purpose
Before evaluating all 40 replies, I test the judge on one ticket. This helps me find problems early and avoid wasting API calls.

### My Approach
The judge checks each reply against four criteria and returns a strict JSON result containing `PASS` or `FAIL` for each criterion.

The judge receives:
- **Ticket:** The original customer issue.
- **Rubric:** The four evaluation criteria.
- **Rulebook:** The bank's approved rules.
- **Answer:** The reply being evaluated.

These sections are enclosed in tags to separate the data from the judge's instructions and reduce the risk of prompt injection.

### What the Code Does
1. `judge_messages(ticket, truth, answer)` prepares the judge's prompts and escapes tags in the input.
2. `lab.chat(...)` sends the prompts to the model.
3. `parse_verdict(text)` checks the response and converts the JSON verdicts into `True` or `False`. It raises `JudgeParseError` if the output has missing keys, extra keys, or invalid values.

### My Observation
I test one reply first to check whether the judge returns valid results. Once it works correctly, I can use it to evaluate more replies.

In [ ]:
test_ticket, test_truth = GOLDEN20[0]
print(f"ticket:  {test_ticket}")
print(f"truth:   {test_truth}")
print(f"answer:  {SAMPLE_REPLY_A}\n")

if ready:
    verdict = judge_answer(lab, test_ticket, test_truth, SAMPLE_REPLY_A)
    print("raw judge reply:", verdict["raw"])
    print("parsed verdicts:", verdict["verdicts"])
    print("all four passed:", verdict["all_pass"])
    if verdict["error"]:
        print("PARSE ERROR:", verdict["error"])
    print(meter.report())
else:
    print("skipped (no key in .env)")

### Expected Output

The judge returns a JSON response with four verdicts. The parser converts them into Python Boolean values (`True` or `False`) and checks whether all four criteria passed.

### Common Issues

| Issue | What I do |
|---|---|
| JSON is missing | Run it again. If it continues, review the judge prompt. |
| Wrong or missing keys | Check the JSON keys and judge instructions. |
| A reply fails a criterion | Review that criterion before deciding whether the judge is correct. |

### My Observation

The judge can make mistakes, so I should not accept every verdict without checking it. In Section 7, I will compare the judge's decisions with my own review.

### Understanding `parse_verdict`

I also test a small version of `parse_verdict` to check that it correctly identifies `PASS` and `FAIL`. This helps me catch parsing mistakes that could affect the final evaluation results.

In [ ]:
import json, re

def my_parse_verdict(text, ids=("c1", "c2", "c3", "c4")):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    obj = json.loads(m.group(0))
    out = {}
    for i in ids:
        if i not in obj:
            raise ValueError(f"missing key {i}")
        v = str(obj[i]).strip().upper()
        if v not in ("PASS", "FAIL"):
            raise ValueError(f"{i} is not PASS/FAIL: {obj[i]!r}")
        out[i] = v == "PASS"
    return out

sample = 'Sure, here you go:\n{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}\nhope that helps!'
mine, real = my_parse_verdict(sample), parse_verdict(sample)
print("mine:", mine)
print("real:", real)
assert mine == real == {"c1": True, "c2": False, "c3": True, "c4": True}
print("match: True")

### Expected Output

My implementation and the original `parse_verdict` should return the same results. The output should show `match: True`.

This test does not require a model or API key.

### My Observation

Checking for extra keys is important because the judge might return unexpected criteria or an incorrect JSON structure. The parser should accept only the four expected keys: `c1`, `c2`, `c3`, and `c4`.

The judge must return structured `PASS` or `FAIL` verdicts. If I need explanations, I can add a separate field to the JSON instead of replacing the required verdicts.

## 5. Evaluating the Baseline Prompt (v1)

### Purpose
I evaluate the baseline prompt using all 20 golden-set tickets. Each answer is checked by the LLM judge against the four rubric criteria.

### My Approach
The `score_prompt` function automates the evaluation:

1. `v1_messages(ticket)` creates the baseline prompt using the original Session 14 format.
2. `score_prompt(lab, lab, v1_messages, GOLDEN20)` generates a reply for each ticket, evaluates it, and counts the passes for each criterion.
3. The same `lab` is used for both generating and judging the replies.

### My Observation
This step evaluates the baseline prompt across all 20 tickets. The results will help me understand its strengths and weaknesses before comparing it with the structured prompt (v2).

**Note:** This step makes 40 model calls: 20 to generate replies and 20 to judge them.

In [ ]:
if ready:
    v1_result = score_prompt(lab, lab, v1_messages, GOLDEN20)
    for c in RUBRIC:
        print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n'])}")
    print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n'])}")
    print(f"graded: {v1_result['graded']}/{v1_result['n']}")
    print(meter.report())
else:
    print("skipped (no key in .env)")
    v1_result = None

### Expected Output

The results show the percentage of replies that pass each rubric criterion and the percentage that pass all four criteria. The example values below are illustrative; my actual results may differ.

### Common Issues

| Issue | What I do |
|---|---|
| Fewer than 20 tickets graded | Check for parsing errors and rerun if several tickets failed. |
| The run is slow | Wait for the model calls to finish. |
| No replies pass all four criteria | Review the individual criteria to understand where the baseline fails. |

### My Observation

A prompt may predict the correct label but still fail to follow the required format, cite the correct rule, or avoid inventing policies. Passing all four criteria is harder because the reply must meet every requirement at the same time.

## 6. Comparing Baseline (v1) and Structured Prompt (v2)

### Purpose
I compare both prompts using the same 20 golden-set tickets and the same LLM judge.

### My Approach
1. `v2_maker` loads my Session 16 prompt files and prepares the structured prompt for each ticket.
2. `compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN20)` evaluates both prompts and returns their scores along with the evaluation order.
3. The **order guard** changes which prompt is evaluated first for each ticket, helping reduce position bias.

The structured prompt is loaded from my own repository, so the evaluation uses the files currently saved there.

### Common Issues

If `WORK_REPO` is not configured or the required prompt files are missing, the code raises `PromptFileError`. I need to check the repository path and file locations before continuing.

### My Observation

Both prompts are evaluated on the same tickets using the same judge. This makes it easier to compare their results fairly and understand whether the structured prompt improves reply quality.

In [ ]:
if ready:
    if not WORK_REPO:
        print("skipped: WORK_REPO is not set. Add it to .env or set it in the shell, then restart the kernel.")
        cmp = None
    else:
        v2_maker = lambda t: v2_messages_from_files(WORK_REPO, t)
        cmp = compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN20)
        v1_result, v2_result = cmp["v1"], cmp["v2"]
        print(f"{'Criterion':<28} {'v1 (baseline)':<16} {'v2 (structured)':<16}")
        for c in RUBRIC:
            print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n']):<16} "
                  f"{pct(v2_result['per_criterion'][c['id']], v2_result['n']):<16}")
        print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n']):<16} "
              f"{pct(v2_result['all_pass'], v2_result['n']):<16}")
        print(f"\nticket order was v1-first for {sum(cmp['order'])}/{len(cmp['order'])} tickets (shuffled, not all-v1-first)")
        print(meter.report())
else:
    print("skipped (no key in .env)")
    cmp = None

### Expected Output

The comparison table shows how v1 and v2 perform on each rubric criterion. My actual scores may differ from the example.

### Common Issues

| Issue | What I do |
|---|---|
| Prompt files are missing | Check `WORK_REPO` and the Session 16 file paths. |
| v2 performs worse than expected | Review the failing replies in `cmp['v2']['rows']` to find areas for improvement. |
| The evaluation order looks uneven | Check the recorded order. A deterministic shuffle can still produce an uneven split. |

### My Observation

I compare the scores for all four criteria, not just the overall result. This helps me identify exactly where the structured prompt improves or needs improvement.

The order guard helps reduce position bias by varying which prompt is evaluated first. Without it, I could not rule out the possibility that answer order influenced the judge's decisions.

## 7. Human Spot-Check

### Purpose
I manually review five tickets to check how closely my judgments match the LLM judge's results.

### My Approach
1. `spot_check_items()` selects the same five tickets each time.
2. The code displays each ticket, the structured prompt's reply, and the judge's verdict.
3. I fill in `MY_VERDICTS` with my own `PASS` or `FAIL` decision for each of the four criteria.
4. `agreement_rate(pairs)` compares my judgments with the judge's results across 20 comparisons.

### My Observation
The human spot-check helps me identify disagreements between my judgments and the LLM judge. It does not prove the judge is always correct, but it gives me a way to check its reliability on a small sample.

In [ ]:
spot = spot_check_items()
v2_rows_by_ticket = {r["ticket"]: r for r in cmp["v2"]["rows"]} if cmp else {}

for i, (ticket, truth) in enumerate(spot, 1):
    row = v2_rows_by_ticket.get(ticket)
    print(f"[{i}] truth={truth}")
    print(f"    ticket: {ticket}")
    if row:
        print(f"    v2 answer: {row['answer']}")
        print(f"    judge said: {row['verdicts']}")
    else:
        print("    (run section 6 first to have an answer + judge verdict to compare against)")
    print()

# YOUR TURN: grade these five yourself, True = PASS, False = FAIL, using the section 3 rubric
MY_VERDICTS = [
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 1 - edit these five dicts to your own judgement
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 2
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 3
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 4
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 5
]

if cmp:
    judge_verdicts = [v2_rows_by_ticket[t]["verdicts"] for t, _ in spot]
    pairs = list(zip(MY_VERDICTS, judge_verdicts))
    spot_result = agreement_rate(pairs)
    print("agreement with the judge:", spot_result)
else:
    spot_result = None
    print("skipped: run section 6 first")

### Expected Output

The notebook displays five tickets and the judge's verdicts. After I enter my own judgments in `MY_VERDICTS`, it calculates the agreement rate.

For example, `18/20 = 90%` means my judgments match the judge on 18 out of 20 criteria. My actual result may differ.

### Common Issues

| Issue | What I do |
|---|---|
| Agreement is below 70% | Review the disagreements and check whether the rubric is unclear or the judge made a mistake. |
| `KeyError` on `v2_rows_by_ticket` | Run Section 6 first and make sure `cmp` contains the comparison results. |

### My Observation

The agreement rate helps me understand how closely the LLM judge matches human judgment. I should review disagreements instead of relying only on the percentage.

In [ ]:
def my_agreement(human, judge):
    ids = sorted(set(human) & set(judge))
    agree = [i for i in ids if human[i] == judge[i]]
    return {"ids": ids, "agree": agree, "disagree": [i for i in ids if i not in agree], "n": len(ids), "agreed": len(agree)}

h = {"c1": True, "c2": True, "c3": False, "c4": True}
j = {"c1": True, "c2": False, "c3": False, "c4": True}
mine, real = my_agreement(h, j), agreement(h, j)
print("mine:", mine)
print("real:", real)
assert mine == real == {"ids": ["c1", "c2", "c3", "c4"], "agree": ["c1", "c3", "c4"], "disagree": ["c2"], "n": 4, "agreed": 3}
print("match: True")

### Expected Output

My implementation and the original agreement function should return the same result, with `match: True`.

This test does not require a model or API key.

### My Observation

Agreement is measured for each criterion separately. With five tickets and four criteria per ticket, there are 20 comparisons.

This gives me more detail than checking only whether I agree with the judge on each entire ticket. I should also make my own judgments **before** reading the judge's verdict to avoid bias.

## 8. Save the Evaluation Report and Check My Work

### Purpose
I save the evaluation results in `experiments/s17_eval.md` so I can review and share them without running the notebook again.

### My Approach
1. `eval_markdown(v1_result, v2_result, MODEL, MODEL, spot_check=spot_result)` creates a Markdown report with the evaluation results.
2. `save_eval_report(WORK_REPO, text)` saves the report in my repository.
3. `check_s17_work(WORK_REPO)` checks the saved files and verifies that the required work is present.

### My Observation
Saving the report keeps my evaluation results available for future review. Checking the files from disk also helps me find missing or incorrect outputs before running the pytest checkpoint.

In [ ]:
SAVE_REPORT = True    # set False to preview without writing the file

if cmp and spot_result:
    report_text = eval_markdown(cmp["v1"], cmp["v2"], MODEL, MODEL, spot_check=spot_result)
    print(report_text)
    if SAVE_REPORT and WORK_REPO:
        path = save_eval_report(WORK_REPO, report_text)
        print(f"saved to {path}")
        print()
        for r in check_s17_work(WORK_REPO):
            print(f"  {'PASS' if r.passed else 'FAIL'}  {r.name}" + (f"   ({r.detail})" if r.detail else ""))
else:
    print("skipped: run sections 6 and 7 first")

### Expected Output

The notebook displays the evaluation report, saves it to `experiments/s17_eval.md`, and checks whether the required files and results are present.

The checks should confirm:
- The work repository and Session 16 prompt files exist.
- The evaluation report is saved.
- Results for both prompts cover all 20 tickets.
- The human spot-check includes at least five tickets.

### Common Issues

| Issue | What I do |
|---|---|
| Report does not cover 20 tickets | Check `cmp['v1']['graded']` and `cmp['v2']['graded']`, then rerun Section 6 if needed. |
| Spot-check is missing | Complete Section 7 and record five tickets in `MY_VERDICTS`. |
| Prompt files are missing | Check `WORK_REPO` and restore the required files if necessary. |

### My Observation

The checks read the saved report from disk, so they verify what is actually stored rather than relying only on notebook variables.

If I rerun Sections 6 or 7 and get new results, I must rerun Section 8 to update the report.

# Part C: Wrap-Up

## C1. Summary

- A correct label does not always mean a good reply. A rubric helps me check the overall quality.
- The golden set reuses Session 16's 20 held-out tickets to compare both prompts fairly.
- The LLM judge returns four `PASS` or `FAIL` verdicts in JSON format.
- Position, verbosity, and self-preference biases can affect the judge's decisions.
- The order guard helps reduce the effect of answer order.
- A human spot-check helps me compare my judgments with the judge's results.
- I keep the same golden set, rubric, and judge when comparing v1 and v2.

## C2. Key Terms

| Term | Meaning |
|---|---|
| Golden Set | A fixed set of test cases used to compare prompt versions. |
| Rubric | A set of criteria used to evaluate responses. |
| LLM-as-Judge | A model that evaluates another model's response. |
| Strict JSON Verdict | A structured result that code can read and process. |
| Position Bias | A preference based on answer order. |
| Verbosity Bias | A preference for longer responses. |
| Self-Preference Bias | A model's tendency to favour its own style. |
| Order Guard | A method that varies which prompt goes first. |
| Human Spot-Check | Manually reviewing a sample of responses. |
| Pass Rate | The percentage of items that pass a criterion. |

## C3. Interview Practice

### 1. How would you evaluate whether a new prompt is better than the old one?

I would use a fixed golden set and a rubric with clear criteria. I would evaluate both prompts using the same judge and an order guard, then compare their results. I would also manually review a sample to check the judge's reliability.

### 2. What is LLM-as-Judge, and what are its limitations?

LLM-as-Judge uses a model to evaluate another model's responses against a rubric. It can have position bias, verbosity bias, and self-preference bias. I use a human spot-check to identify possible disagreements.

### 3. Why should you shuffle answer order?

Shuffling helps reduce the influence of answer order on evaluation. In this project, the order guard deterministically varies which prompt goes first for each ticket.

### 4. What is the difference between a golden set and a held-out set?

A held-out set is data kept separate from training or few-shot examples. A golden set is a fixed set used to evaluate different prompt versions. In this project, I reuse Session 16's held-out tickets as the golden set.

### 5. How would you check whether an LLM judge is reliable?

I would manually evaluate a fixed sample using the same rubric and compare my judgments with the judge's results. Here, I check five tickets across four criteria, giving 20 comparisons. This provides a useful check but does not guarantee that the judge is always correct.

## C4. Troubleshooting

| Error | Possible cause | What I do |
|---|---|---|
| `ImportError` | Course files or dependencies are outdated. | Update the repository, run `uv sync`, and restart the kernel. |
| `PromptFileError` | The repository path is incorrect or prompt files are missing. | Check `WORK_REPO` and the Session 16 files. |
| `JudgeParseError` | The judge returned invalid JSON. | Review the error and retry if needed. |
| Checkpoint test skipped | API key is missing. | Configure the key in `.env`. |
| Checkpoint test fails | The evaluation report is incomplete. | Fix the failed checks, save the report again, and rerun the test. |
| Results differ from another run | Model responses or judge decisions vary. | Review my own results and investigate significant differences. |

## C5. What Comes Next?

In Session 18, I will test the structured prompt against different prompt-injection attacks. I will reuse the same judge and rubric to check how well the prompt handles these attacks.

I will keep the following files for the next session:

- `prompts/support_classifier/`
- `evaluation_tasks.py`
- `experiments/s17_eval.md`

These files help me continue testing and improving the prompt without rebuilding everything.